# D4 · Lab 2 — Strict Tool Use & Enum Drift

**Proves:** without `strict`, enum values can drift (note the deliberately mixed-case enum); with `strict: True` they are locked; a strict tool and `output_config.format` run together.

Requires `ANTHROPIC_API_KEY` set and `pip install anthropic pydantic`. Run the cells top to bottom. Full write-up is in `README.md` (this folder).


## Setup
The `route_ticket` tool (mixed-case enum is intentional bait), 15 tickets, and the routing schema.


In [ ]:
import os
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY before running (see common/setup-prereq-guide.md)."

from anthropic import Anthropic
from pydantic import BaseModel
import json

client = Anthropic()

TOOL_STRICT = {
    "name": "route_ticket",
    "description": "Routes a support ticket to the correct team based on category.",
    #"strict": True,
    "input_schema": {
        "type": "object",
        "properties": {
            "category": {
                "type": "string",
                "enum": [
                    "Billing",
                    "technical",
                    "Account",
                    "other"
                ],
                "description": "The category of the support ticket."
            },
            "summary": {
                "type": "string",
                "description": "One-sentence summary ofthe issue"
            }
        },
        "required": ["category", "summary"],
        "additionalProperties": False
    }
}

TICKETS = [
    "I was charged twice this month",
    "My login isn't working",
    "Can't access my dashboard after password reset",
    "The API keeps returning 500 errors",
    "Billing statement shows wrong company name",
    "My subscription says active but features are locked",
    "Need to update credit card on file",
    "Account got suspended without warning",
    "The software crashes on startup",
    "I want to cancel my plan",
    "Invoice was sent to wrong email",
    "Two-factor auth is not sending codes",
    "Payment failed but money was deducted",
    "I need to transfer my account to another user",
    "The app is very slow and keeps freezing",   
]

strict_log = []


class RoutingResult(BaseModel):
    routed_to: str
    priority: str    # "high", "medium", "low"
    reason: str
routing_schema = RoutingResult.model_json_schema()
routing_schema["additionalProperties"] = False


### Run: route the tickets
Scan the logged categories for drift (casing mismatches, invented values). Then **uncomment** `"strict": True` in the tool above and re-run this cell to see values locked to the exact enum.


In [ ]:
for ticket in TICKETS:
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        tools=[TOOL_STRICT],
        tool_choice={"type":"auto"},
        output_config={
            "format":{
                "type": "json_schema",
                "schema": routing_schema
            }
        },
        messages=[{"role":"user","content":f"Route this ticket: {ticket}"}]
    )

    for block in response.content:
        if block.type == "tool_use":
            strict_log.append({
                "ticket": ticket,
                "input": block.input
            })

for entry in strict_log:
    print(f"[{entry['input'].get('category')}] {entry['ticket']}")
